In [78]:
# Import libraries and set seed
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
np.random.seed(8)

## Read & Clean Data

In [79]:
# Read data 
menstrual = pd.read_csv('data.csv')
menstrual = menstrual.drop(columns = 'id')
menstrual.head()

,day_in_study,sedentary_mins,active_mins,glucose_mean,rhr_mean,minutes_asleep,minutes_awake,sleep_efficiency_mean,sleep_score_overall,sleep_score_revitalization,...,headaches,cramps,sorebreasts,fatigue,sleepissue,moodswing,stress,foodcravings,indigestion,bloating
0,1,753.0,64,5.498958,74.785346,1027.0,26.0,98.0,NaN,NaN,...,High,Very Low/Little,Very Low/Little,High,Low,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
1,2,855.0,74,5.372222,80.407307,77.0,4.0,95.0,NaN,NaN,...,Very High,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
2,3,751.0,159,5.579514,84.686869,502.0,28.0,95.0,NaN,NaN,...,High,Very Low/Little,Very Low/Little,Very High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
3,4,905.0,86,5.206597,83.852219,384.0,65.0,93.0,80.0,22.0,...,Very Low/Little,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
4,5,1430.0,10,5.381597,0.000000,NaN,NaN,NaN,NaN,NaN,...,Very Low/Little,Very Low/Little,Very Low/Little,High,High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little


In [80]:
mlp_data = menstrual[['day_in_study', 'active_mins', 'glucose_mean', 'rhr_mean', 'sleep_score_overall', 'phase', 'lh', 'estrogen', 'sleep_efficiency_mean', 'headaches', 'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings', 'indigestion', 'bloating']]
mlp_data.head()

,day_in_study,active_mins,glucose_mean,rhr_mean,sleep_score_overall,phase,lh,estrogen,sleep_efficiency_mean,headaches,cramps,sorebreasts,fatigue,sleepissue,moodswing,stress,foodcravings,indigestion,bloating
0,1,64,5.498958,74.785346,NaN,Follicular,2.9,94.2,98.0,High,Very Low/Little,Very Low/Little,High,Low,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
1,2,74,5.372222,80.407307,NaN,Follicular,1.2,226.3,95.0,Very High,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
2,3,159,5.579514,84.686869,NaN,Follicular,3.5,276.8,95.0,High,Very Low/Little,Very Low/Little,Very High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
3,4,86,5.206597,83.852219,80.0,Fertility,1.8,322.1,93.0,Very Low/Little,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
4,5,10,5.381597,0.000000,NaN,Fertility,4.6,244.9,NaN,Very Low/Little,Very Low/Little,Very Low/Little,High,High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little


In [81]:
targets = ['headaches', 'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings', 'indigestion', 'bloating']
mlp_bin = mlp_data.copy()
for col in targets:
    mlp_bin[f'bin_{col}'] = (mlp_data[col] != 'Not at all').astype(int)

mlp_bin = mlp_bin.drop(columns=targets)
mlp_bin.head()

,day_in_study,active_mins,glucose_mean,rhr_mean,sleep_score_overall,phase,lh,estrogen,sleep_efficiency_mean,bin_headaches,bin_cramps,bin_sorebreasts,bin_fatigue,bin_sleepissue,bin_moodswing,bin_stress,bin_foodcravings,bin_indigestion,bin_bloating
0,1,64,5.498958,74.785346,NaN,Follicular,2.9,94.2,98.0,1,1,1,1,1,1,1,1,1,1
1,2,74,5.372222,80.407307,NaN,Follicular,1.2,226.3,95.0,1,1,1,1,1,1,1,1,1,1
2,3,159,5.579514,84.686869,NaN,Follicular,3.5,276.8,95.0,1,1,1,1,1,1,1,1,1,1
3,4,86,5.206597,83.852219,80.0,Fertility,1.8,322.1,93.0,1,1,1,1,1,1,1,1,1,1
4,5,10,5.381597,0.000000,NaN,Fertility,4.6,244.9,NaN,1,1,1,1,1,1,1,1,1,1


In [82]:
mlp_onehot = pd.get_dummies(mlp_bin, drop_first = True, dtype = int)
mlp_onehot.head()

,day_in_study,active_mins,glucose_mean,rhr_mean,sleep_score_overall,lh,estrogen,sleep_efficiency_mean,bin_headaches,bin_cramps,...,bin_fatigue,bin_sleepissue,bin_moodswing,bin_stress,bin_foodcravings,bin_indigestion,bin_bloating,phase_Follicular,phase_Luteal,phase_Menstrual
0,1,64,5.498958,74.785346,NaN,2.9,94.2,98.0,1,1,...,1,1,1,1,1,1,1,1,0,0
1,2,74,5.372222,80.407307,NaN,1.2,226.3,95.0,1,1,...,1,1,1,1,1,1,1,1,0,0
2,3,159,5.579514,84.686869,NaN,3.5,276.8,95.0,1,1,...,1,1,1,1,1,1,1,1,0,0
3,4,86,5.206597,83.852219,80.0,1.8,322.1,93.0,1,1,...,1,1,1,1,1,1,1,0,0,0
4,5,10,5.381597,0.000000,NaN,4.6,244.9,NaN,1,1,...,1,1,1,1,1,1,1,0,0,0


In [83]:
print(mlp_onehot.isna().sum())

day_in_study               0
active_mins                0
glucose_mean             590
rhr_mean                   0
sleep_score_overall      450
lh                       223
estrogen                 224
sleep_efficiency_mean    526
bin_headaches              0
bin_cramps                 0
bin_sorebreasts            0
bin_fatigue                0
bin_sleepissue             0
bin_moodswing              0
bin_stress                 0
bin_foodcravings           0
bin_indigestion            0
bin_bloating               0
phase_Follicular           0
phase_Luteal               0
phase_Menstrual            0
dtype: int64


In [84]:
# Impute missing features with mean
mlp_onehot['glucose_mean'] = mlp_onehot['glucose_mean'].fillna(mlp_onehot['glucose_mean'].mean())
mlp_onehot['sleep_score_overall'] = mlp_onehot['sleep_score_overall'].fillna(mlp_onehot['glucose_mean'].mean())
mlp_onehot['lh'] = mlp_onehot['lh'].fillna(mlp_onehot['lh'].mean())
mlp_onehot['estrogen'] = mlp_onehot['estrogen'].fillna(mlp_onehot['estrogen'].mean())
mlp_onehot['sleep_efficiency_mean'] = mlp_onehot['sleep_efficiency_mean'].fillna(mlp_onehot['sleep_efficiency_mean'].mean())

print(mlp_onehot.isna().sum())

day_in_study             0
active_mins              0
glucose_mean             0
rhr_mean                 0
sleep_score_overall      0
lh                       0
estrogen                 0
sleep_efficiency_mean    0
bin_headaches            0
bin_cramps               0
bin_sorebreasts          0
bin_fatigue              0
bin_sleepissue           0
bin_moodswing            0
bin_stress               0
bin_foodcravings         0
bin_indigestion          0
bin_bloating             0
phase_Follicular         0
phase_Luteal             0
phase_Menstrual          0
dtype: int64


In [85]:
target_cols = [col for col in mlp_onehot.columns if 'bin' in col]
for col in target_cols:
    mlp_onehot[col] = mlp_onehot.pop(col)

mlp_onehot.head()

,day_in_study,active_mins,glucose_mean,rhr_mean,sleep_score_overall,lh,estrogen,sleep_efficiency_mean,phase_Follicular,phase_Luteal,...,bin_headaches,bin_cramps,bin_sorebreasts,bin_fatigue,bin_sleepissue,bin_moodswing,bin_stress,bin_foodcravings,bin_indigestion,bin_bloating
0,1,64,5.498958,74.785346,11.406948,2.9,94.2,98.000000,1,0,...,1,1,1,1,1,1,1,1,1,1
1,2,74,5.372222,80.407307,11.406948,1.2,226.3,95.000000,1,0,...,1,1,1,1,1,1,1,1,1,1
2,3,159,5.579514,84.686869,11.406948,3.5,276.8,95.000000,1,0,...,1,1,1,1,1,1,1,1,1,1
3,4,86,5.206597,83.852219,80.000000,1.8,322.1,93.000000,0,0,...,1,1,1,1,1,1,1,1,1,1
4,5,10,5.381597,0.000000,11.406948,4.6,244.9,92.455916,0,0,...,1,1,1,1,1,1,1,1,1,1


## Prep Data for MLP

In [86]:
mlp_array = mlp_onehot.to_numpy()
print(mlp_array)

[[  1.          64.           5.49895833 ...   1.           1.
    1.        ]
 [  2.          74.           5.37222222 ...   1.           1.
    1.        ]
 [  3.         159.           5.57951389 ...   1.           1.
    1.        ]
 ...
 [ 88.         181.           7.15138889 ...   1.           1.
    1.        ]
 [ 89.         267.           7.11875    ...   1.           1.
    1.        ]
 [ 90.         331.          11.40694841 ...   1.           1.
    1.        ]]


In [87]:
# MinMax Scaling
X = mlp_array
scaler = MinMaxScaler()
X_scale = scaler.fit_transform(X).round(2)
X_scale

array([[0.  , 0.08, 0.02, ..., 1.  , 1.  , 1.  ],
       [0.01, 0.09, 0.02, ..., 1.  , 1.  , 1.  ],
       [0.02, 0.2 , 0.02, ..., 1.  , 1.  , 1.  ],
       ...,
       [0.98, 0.23, 0.03, ..., 1.  , 1.  , 1.  ],
       [0.99, 0.34, 0.03, ..., 1.  , 1.  , 1.  ],
       [1.  , 0.42, 0.06, ..., 1.  , 1.  , 1.  ]])

In [88]:
# Separate into Phi and y, Add Bias Term, and Train-Test-Split 
Phi = X_scale[:,:-10]
ones = np.ones((Phi.shape[0], 1))
Phi = np.hstack((Phi, ones))

y = X_scale[:,-10:]

Phi_train, Phi_test, y_train, y_test = train_test_split(Phi, y, test_size = .3)

In [89]:
# Inspect training set
Phi_train, y_train

(array([[0.33, 0.24, 0.06, ..., 0.  , 1.  , 1.  ],
        [0.63, 0.1 , 0.02, ..., 0.  , 1.  , 1.  ],
        [0.03, 0.44, 0.02, ..., 0.  , 0.  , 1.  ],
        ...,
        [0.6 , 0.42, 0.06, ..., 0.  , 1.  , 1.  ],
        [0.16, 0.41, 0.02, ..., 1.  , 0.  , 1.  ],
        [0.01, 0.16, 0.02, ..., 0.  , 1.  , 1.  ]]),
 array([[1., 0., 1., ..., 1., 1., 1.],
        [1., 1., 0., ..., 1., 1., 1.],
        [1., 0., 0., ..., 0., 0., 0.],
        ...,
        [1., 1., 1., ..., 1., 1., 1.],
        [1., 1., 1., ..., 1., 0., 1.],
        [1., 1., 1., ..., 1., 1., 1.]]))

## Implement MLP From Scratch Using Gradient Descent

In [90]:
# Define step size (eta), epochs, number of hidden nodes (q), W1, W2
eta = 0.3
epochs = 1000

q = 4
d = Phi_train.shape[1]

W1 = np.random.randn(d,q)
W2 = np.random.randn(q,1)

In [91]:
# Define f(x) using ReLU for h and sigmoid for output
def f(x):
    h = np.maximum(0, W1.T @ x)
    return 1 / (1 + np.exp(-1 * (W2.T @ h)))

In [92]:
# Run Gradient Descent
errors = []
n = Phi_train.shape[0]

for epoch in range(epochs):
    # calculate dW2
    dW2 = 0
    for i, j in enumerate(y_train):
        x = np.reshape(Phi_train[i], (d, 1))
        h = np.maximum(0, W1.T @ x)
        dW2 += (1 / n) * (f(x) - y_train[i]) * h

    # calculate dW1
    dW1 = 0
    for i, j in enumerate(y_train):
        x = np.reshape(Phi_train[i], (d, 1))
        h = np.maximum(0, W1.T @ x)
        mat1 = np.heaviside(h, 0)
        dW1 += (1 / n) * (f(x) - y_train[i]) * np.kron(x, (W2 * mat1).T)

    # update W1 & W2
    W1 = W1 - eta * dW1
    W2 = W2 - eta * dW2

    # calculate f(x) at each training value and error (log loss)
    f_x = []
    for i, j in enumerate(Phi_train):
         x = np.reshape(Phi_train[i], (d, 1))
         f_x.append(f(x))
         
    f_x = np.array(f_x).flatten()
    
    e = (1 / n) * np.sum((-y_train * np.log(f_x)) - ((1 - y_train) * np.log(1 - f_x)))
    errors.append(e)

ValueError: operands could not be broadcast together with shapes (1,10) (12,4) 

In [ ]:
# Plot convergence
import matplotlib.pyplot as plt
plt.plot(range(epochs), errors, label='line')
plt.show()

In [ ]:
# Print final estimates of W1 and W2
print('W1:\n', W1)
print('W2:\n', W2)

In [ ]:
# Predict using test data
y_pred = []
for i, j in enumerate(Phi_test):
    x = np.reshape(Phi_test[i], (d, 1))
    y_pred.append(f(x))

y_pred = np.array(y_pred).flatten()
print(y_pred)

In [ ]:
# Evaluate accuracy of algorithm using log loss and accuracy
n_test = Phi_test.shape[0]
log_loss = (1 / n_test) * np.sum((-y_test * np.log(y_pred)) - ((1 - y_test) * np.log(1 - y_pred)))

y_pred_binary = np.round(y_pred).flatten()
matches = 0
for i in range(len(y_pred_binary)):
    if y_pred_binary[i] == y_test[i]:
        matches += 1

print('Log Loss:', log_loss)
print('Accuracy:', matches / n_test)